# 05 — Feature Engineering

**Proyecto:** RepostaPro — Optimización del repostaje en flotas comerciales  
**Autor:** Víctor González Martín  
**Notebook:** 05 — Generación de features para modelado predictivo 

## Objetivo del notebook

Aplicar la lógica de feature engineering definida en `src/features.py` sobre el dataset consolidado. El resultado son dos datasets analíticos especializados (uno por carburante del núcleo predictivo) listos para la fase de modelado.

## Carburantes objetivo

Limitamos el modelado a los dos carburantes con cobertura nacional > 95 %:

- **Gasóleo A**: 98,1 % de cobertura.
- **Gasolina 95 E5**: 95,0 % de cobertura.

Los premium (Gasóleo Premium 52 %, Gasolina 98 E5 48 %) se descartan por menor representatividad y cobertura sesgada. Permanecen en el análisis descriptivo de la memoria.

## Features generadas

15 nuevas variables agrupadas en 5 familias:

| Familia | Variables |
|---|---|
| Calendario | `dia_semana`, `mes`, `dia_del_mes`, `semana_del_año`, `es_fin_semana`, `es_festivo_nacional` |
| Régimen geopolítico | `regimen`, `dias_desde_shock` |
| Lags temporales | `precio_lag_1`, `precio_lag_7`, `precio_lag_30` |
| Medias móviles | `precio_mm_7`, `precio_mm_30` |
| Contexto nacional | `precio_medio_nacional_dia`, `diferencial_vs_nacional` |

In [1]:
# Configuración del entorno
import sys
import warnings
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["font.family"] = "Calibri"
plt.rcParams["font.size"] = 11
sns.set_style("whitegrid")
warnings.filterwarnings("ignore", category=FutureWarning)

# Añadir la raíz del proyecto al sys.path
RAIZ_PROYECTO = Path("..").resolve()
if str(RAIZ_PROYECTO) not in sys.path:
    sys.path.insert(0, str(RAIZ_PROYECTO))

# Importar funciones del módulo de features
from src.features import (preparar_dataset_carburante,FECHA_CORTE_SHOCK,CARBURANTES_NUCLEO,)

print(f"Raíz del proyecto: {RAIZ_PROYECTO}")
print(f"Fecha de corte del shock: {FECHA_CORTE_SHOCK.date()}")
print(f"Carburantes núcleo: {CARBURANTES_NUCLEO}")

Raíz del proyecto: C:\TFM
Fecha de corte del shock: 2026-03-01
Carburantes núcleo: ['Precio Gasoleo A', 'Precio Gasolina 95 E5']


## Carga del histórico consolidado

Cargamos los 3 Parquet anuales generados y los concatenamos en un único DataFrame.

In [2]:
# Cargar el histórico completo
CARPETA_PROCESSED = Path("../data/processed")
CARPETA_FEATURES = Path("../data/processed/features")
CARPETA_FEATURES.mkdir(parents=True, exist_ok=True)

print("Cargando los 3 Parquet anuales...")
ficheros_parquet = sorted(CARPETA_PROCESSED.glob("historico_carburantes_*.parquet"))
df = pd.concat([pd.read_parquet(f) for f in ficheros_parquet], ignore_index=True)

print(f"\n Dataset histórico cargado")
print(f"  Filas:    {len(df):,}")
print(f"  Columnas: {df.shape[1]}")
print(f"  Memoria:  {df.memory_usage(deep=True).sum() / 1024 / 1024 / 1024:.2f} GB")
print(f"  Rango:    {df['fecha'].min().date()} → {df['fecha'].max().date()}")

Cargando los 3 Parquet anuales...

✓ Dataset histórico cargado
  Filas:    10,040,121
  Columnas: 25
  Memoria:  2.95 GB
  Rango:    2024-01-01 → 2026-06-14


## Prueba unitaria: aplicación de features a Gasóleo A

Antes de procesar el dataset completo, validamos que el módulo funciona correctamente con una muestra reducida (un solo mes de datos). Esto permite detectar errores sin esperar el procesado completo.

In [3]:
# Prueba unitaria con una muestra
muestra = df[(df["fecha"] >= "2026-04-01") & (df["fecha"] <= "2026-04-30")].copy()

print(f"Muestra de prueba: abril 2026 ({len(muestra):,} filas)\n")

muestra_features = preparar_dataset_carburante(muestra, "Precio Gasoleo A")

print(f"\n=== Validación de features generadas ===")
print(f"\nColumnas del dataset resultante ({muestra_features.shape[1]}):")
for c in muestra_features.columns:
    print(f"  - {c}")

print(f"\nPrimeras 3 filas (columnas seleccionadas):")
cols_muestra = ["IDEESS", "fecha", "precio", "dia_semana", "es_fin_semana","regimen", "precio_lag_1", "precio_lag_7","precio_mm_7", "precio_medio_nacional_dia"]
print(muestra_features[cols_muestra].head(3).to_string())

Muestra de prueba: abril 2026 (344,645 filas)

Preparando dataset para: Precio Gasoleo A
  Filas de entrada:      344,645
  Filas tras filtrado:   338,260 (cobertura: 98.1%)
  ✓ Familia 1 (calendario) aplicada
  ✓ Familia 2 (régimen) aplicada
  ✓ Familia 3 (lags) aplicada
  ✓ Familia 4 (medias móviles) aplicada
  ✓ Familia 5 (contexto nacional) aplicada
  Filas finales:         338,260
  Columnas finales:      40

=== Validación de features generadas ===

Columnas del dataset resultante (40):
  - IDEESS
  - IDMunicipio
  - IDProvincia
  - IDCCAA
  - C.P.
  - Dirección
  - Localidad
  - Municipio
  - Provincia
  - Latitud
  - Longitud (WGS84)
  - Rótulo
  - Horario
  - Tipo Venta
  - precio
  - Precio Gasoleo Premium
  - Precio Gasolina 95 E5
  - Precio Gasolina 98 E5
  - Precio Gases licuados del petróleo
  - Precio Adblue
  - % BioEtanol
  - % Éster metílico
  - Rotulo_normalizado
  - outlier_geografico
  - fecha
  - dia_semana
  - mes
  - dia_del_mes
  - semana_del_año
  - es_fin_sem

In [5]:
# Investigación: ¿por qué precio_mm_7 está NaN en las primeras filas?
# Vamos a ver las primeras 10 filas de una estación concreta tras el procesado
muestra_estacion = muestra_features[muestra_features["IDEESS"] == muestra_features["IDEESS"].iloc[0]].sort_values("fecha").head(10)

print("Primeras 10 filas de la primera estación tras el procesado:\n")
print(muestra_estacion[["fecha", "precio", "precio_lag_1", "precio_mm_7", "precio_mm_30"]].to_string())

# Conteo de NaN en cada columna de features
print("\n\nNaN por columna de features:")
features_a_revisar = ["precio_lag_1", "precio_lag_7", "precio_lag_30","precio_mm_7", "precio_mm_30"]
for col in features_a_revisar:
    n_nan = muestra_features[col].isna().sum()
    total = len(muestra_features)
    print(f"  {col}: {n_nan:,} / {total:,} ({n_nan/total*100:.1f}%)")

Primeras 10 filas de la primera estación tras el procesado:

       fecha  precio  precio_lag_1  precio_mm_7  precio_mm_30
0 2026-04-01   1.809           NaN          NaN           NaN
1 2026-04-02   1.809         1.809     1.809000      1.809000
2 2026-04-03   1.809         1.809     1.809000      1.809000
3 2026-04-04   1.809         1.809     1.809000      1.809000
4 2026-04-05   1.809         1.809     1.809000      1.809000
5 2026-04-06   1.859         1.809     1.809000      1.809000
6 2026-04-07   1.859         1.859     1.817333      1.817333
7 2026-04-08   1.909         1.859     1.823286      1.823286
8 2026-04-09   1.909         1.909     1.837571      1.834000
9 2026-04-10   1.909         1.909     1.851857      1.842333


NaN por columna de features:
  precio_lag_1: 11,366 / 338,260 (3.4%)
  precio_lag_7: 79,502 / 338,260 (23.5%)
  precio_lag_30: 338,260 / 338,260 (100.0%)
  precio_mm_7: 11,366 / 338,260 (3.4%)
  precio_mm_30: 11,366 / 338,260 (3.4%)


## Procesado completo: Gasóleo A

Aplicamos el módulo al histórico completo para Gasóleo A. 

In [6]:
# Procesado completo: Gasóleo A
import time

print(">>> Procesando Gasóleo A...\n")
inicio = time.time()

df_gasoleo = preparar_dataset_carburante(df, "Precio Gasoleo A")

duracion = time.time() - inicio
print(f"\n Procesado completado en {duracion:.1f}s ({duracion/60:.1f} min)")

# Guardar en disco
ruta_gasoleo = CARPETA_FEATURES / "features_gasoleo_a.parquet"
df_gasoleo.to_parquet(ruta_gasoleo, compression="snappy", index=False)

tamano_mb = ruta_gasoleo.stat().st_size / 1024 / 1024
print(f"  Guardado en: {ruta_gasoleo}")
print(f"  Tamaño:      {tamano_mb:.1f} MB")

>>> Procesando Gasóleo A...

Preparando dataset para: Precio Gasoleo A
  Filas de entrada:      10,040,121
  Filas tras filtrado:   9,858,873 (cobertura: 98.2%)
  ✓ Familia 1 (calendario) aplicada
  ✓ Familia 2 (régimen) aplicada
  ✓ Familia 3 (lags) aplicada
  ✓ Familia 4 (medias móviles) aplicada
  ✓ Familia 5 (contexto nacional) aplicada
  Filas finales:         9,858,873
  Columnas finales:      40

✓ Procesado completado en 44.4s (0.7 min)
  Guardado en: ..\data\processed\features\features_gasoleo_a.parquet
  Tamaño:      137.7 MB


## Validación: distribución por régimen y carburante

Verificamos que las features se han generado correctamente examinando algunos estadísticos clave por régimen.

In [8]:
# Validación: estadísticas por régimen
print("ESTADÍSTICAS DE GASÓLEO A POR RÉGIMEN")
print("=" * 60)

por_regimen = df_gasoleo.groupby("regimen").agg(n_filas=("precio", "size"),
    precio_medio=("precio", "mean"),
    precio_std=("precio", "std"),
    precio_min=("precio", "min"),
    precio_max=("precio", "max"),
    estaciones_unicas=("IDEESS", "nunique"),
    dias_unicos=("fecha", "nunique"),).round(4)

print(por_regimen.to_string())
print("\n\nDISTRIBUCIÓN DE NaN POR FEATURE (esperable en lags y medias móviles)")
print("=" * 60)
nans_por_col = df_gasoleo.isna().sum()
nans_relevantes = nans_por_col[nans_por_col > 0].sort_values(ascending=False)
print(nans_relevantes.to_string())

print("\n\nVALIDACIÓN DE NO-DATA-LEAKAGE EN MEDIAS MÓVILES")
print("=" * 60)
# La media móvil del día t NO debe coincidir con el precio del día t
muestra_estacion = df_gasoleo[df_gasoleo["IDEESS"] == df_gasoleo["IDEESS"].iloc[0]].sort_values("fecha")
print(muestra_estacion[["fecha", "precio", "precio_lag_1", "precio_mm_7"]].head(15).to_string())

ESTADÍSTICAS DE GASÓLEO A POR RÉGIMEN
            n_filas  precio_medio  precio_std  precio_min  precio_max  estaciones_unicas  dias_unicos
regimen                                                                                              
post_shock  1192741        1.7313      0.1392       0.945       2.389              11429          106
pre_shock   8666132        1.4415      0.1191       0.849       2.259              11622          787


DISTRIBUCIÓN DE NaN POR FEATURE (esperable en lags y medias móviles)
Precio Gases licuados del petróleo    9041355
Precio Adblue                         8835216
Precio Gasolina 98 E5                 4842135
Precio Gasoleo Premium                3938479
Precio Gasolina 95 E5                  351055
precio_lag_30                          350092
precio_lag_7                            81815
precio_lag_1                            11690
precio_mm_7                             11690
precio_mm_30                            11690


VALIDACIÓN DE NO-DATA

## Procesado de Gasolina 95 E5

Aplicamos el mismo procesado al segundo carburante del núcleo predictivo.

In [9]:
# Procesado completo: Gasolina 95 E5
print(">>> Procesando Gasolina 95 E5...\n")
inicio = time.time()

df_gasolina = preparar_dataset_carburante(df, "Precio Gasolina 95 E5")

duracion = time.time() - inicio
print(f"\n Procesado completado en {duracion:.1f}s ({duracion/60:.1f} min)")

# Guardar en disco
ruta_gasolina = CARPETA_FEATURES / "features_gasolina_95.parquet"
df_gasolina.to_parquet(ruta_gasolina, compression="snappy", index=False)

tamano_mb = ruta_gasolina.stat().st_size / 1024 / 1024
print(f"  Guardado en: {ruta_gasolina}")
print(f"  Tamaño:      {tamano_mb:.1f} MB")

>>> Procesando Gasolina 95 E5...

Preparando dataset para: Precio Gasolina 95 E5
  Filas de entrada:      10,040,121
  Filas tras filtrado:   9,529,417 (cobertura: 94.9%)
  ✓ Familia 1 (calendario) aplicada
  ✓ Familia 2 (régimen) aplicada
  ✓ Familia 3 (lags) aplicada
  ✓ Familia 4 (medias móviles) aplicada
  ✓ Familia 5 (contexto nacional) aplicada
  Filas finales:         9,529,417
  Columnas finales:      40

✓ Procesado completado en 60.3s (1.0 min)
  Guardado en: ..\data\processed\features\features_gasolina_95.parquet
  Tamaño:      128.5 MB


## Resumen

Recapitulación de los dos datasets analíticos generados.

In [10]:
# Resumen 
print("=" * 70)
print("RESUMEN")
print("=" * 70)

datasets = {"Gasóleo A": (df_gasoleo, ruta_gasoleo),"Gasolina 95 E5": (df_gasolina, ruta_gasolina),}

for nombre, (df_x, ruta) in datasets.items():
    print(f"\n>>> {nombre}")
    print(f"    Filas:               {len(df_x):,}")
    print(f"    Columnas:            {df_x.shape[1]}")
    print(f"    Estaciones únicas:   {df_x['IDEESS'].nunique():,}")
    print(f"    Días únicos:         {df_x['fecha'].nunique():,}")
    print(f"    Fichero:             {ruta.name} ({ruta.stat().st_size / 1024 / 1024:.1f} MB)")
    print(f"    Régimenes:")
    for reg, sub in df_x.groupby("regimen"):
        print(f"      - {reg}: {len(sub):,} filas, {sub['fecha'].nunique()} días")

print("\n" + "=" * 70)
print("Feature Engineering completado.")
print("=" * 70)

RESUMEN — Sprint 3 Fase 2: Feature Engineering

>>> Gasóleo A
    Filas:               9,858,873
    Columnas:            40
    Estaciones únicas:   11,690
    Días únicos:         893
    Fichero:             features_gasoleo_a.parquet (137.7 MB)
    Régimenes:
      - post_shock: 1,192,741 filas, 106 días
      - pre_shock: 8,666,132 filas, 787 días

>>> Gasolina 95 E5
    Filas:               9,529,417
    Columnas:            40
    Estaciones únicas:   11,291
    Días únicos:         893
    Fichero:             features_gasolina_95.parquet (128.5 MB)
    Régimenes:
      - post_shock: 1,154,006 filas, 106 días
      - pre_shock: 8,375,411 filas, 787 días

✓ Feature Engineering completado. Listo para Sprint 3 Fase 3 (partición).
